> 在线讲义：[第 11 章 · 大写的 Q、K、V：一次算出所有分数](https://wikieden.github.io/transformer-visual/lessons/11-matrix-qk.html) · 本书《从零读懂 Transformer》

# 第 11 章 · 大写的 Q、K、V：一次算出所有分数

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/11-matrix-qk.html) · 仓库里 `transformer-deep-dive/notes/lessons/11-matrix-qk.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 大写 Q | 每一行是一个词的 q | `Q.shape` | (3, 2)：n = 3、$d_k$ = 2 |
| 一格一格算 | 第 i 行第 j 列 = $\mathbf q_i \cdot \mathbf k_j$ | 两层 for 循环，9 次点积 | 3 × 3 分数表 |
| 一次算完 | $QK^\top$ | `Q @ K.T` | [[1, 0, 1], [0, 2, 1], [0, 4, 2]]，和循环一样 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
WORDS = ["小明", "小红", "她"]
Q = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.0, 2.0]])  # 每一行是一个词的 q
K = torch.tensor([[1.0, 0.0], [0.0, 2.0], [1.0, 1.0]])  # 每一行是一个词的 k

**函数 `score_table_by_loops`**：第 i 行第 j 列 = 第 i 个词的 q · 第 j 个词的 k，一格一格算。

In [ ]:
def score_table_by_loops(q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """第 i 行第 j 列 = 第 i 个词的 q · 第 j 个词的 k，一格一格算。"""
    n = q.shape[0]
    table = torch.zeros(n, n)
    for i in range(n):
        for j in range(n):
            table[i, j] = q[i] @ k[j]
    return table

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("Q 形状:", tuple(Q.shape), "（n = 3 个词，d_k = 2）")
loops = score_table_by_loops(Q, K)
once = Q @ K.T
print("一格一格算（9 次点积）:\n", loops)
print("Q @ K.T 一次算完:\n", once)
print("一样吗:", torch.equal(loops, once))
print("行 = 谁去找，列 = 被找的是谁；'她'那一行:", once[2].tolist())

## 3. 代码里才看得到的

1. **循环和矩阵乘法结果一模一样，但速度差很多**：循环要 Python 一格一格地算；`Q @ K.T` 一次交给底层，GPU 上几千格同时算。这就是"排成矩阵"的意义。
2. **`K.T` 是转置**：先照着用，第 12 章讲为什么要转。

## 4. 自己改一改

- 把上面任意一格里的数字改一改，再按 Shift + Enter，看结果怎么变。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 小写一个词，大写全部词，一行一个词；
> $QK^\top$ 是分数表：行是谁去找，列是被找的，格子是 q · k；排成矩阵一次算完。